In [ ]:
import pandas as pd
import numpy as np 
from scipy.stats import multivariate_normal



## Scenario Generation Function

Generate X scenarios of Y weeks of future stock price growth rates using multivariate normal distribution.

In [ ]:
def generate_scenarios(
    n_scenarios: int,
    weeks_ahead: int,
    expected_returns: list,
    cov_matrix: pd.DataFrame,
    random_seed: int = None
) -> dict:
    """
    Generate scenarios of future stock price growth rates.
    
    Each scenario produces weeks_ahead weeks of per-week simple returns
    for each stock, sampled from a multivariate normal distribution.
    
    Args:
        n_scenarios: Number of scenarios to generate (X)
        weeks_ahead: Number of weeks to project forward (Y)
        expected_returns: List of geometric returns (mean of distribution)
        cov_matrix: Covariance matrix (DataFrame or array) of stock returns
        random_seed: Optional seed for reproducibility in testing
    
    Returns:
        Dict: {scenario_0: df, scenario_1: df, ...} where each df has
              shape (weeks_ahead, n_tickers) with per-week simple returns
    """
    # Validate inputs
    if len(expected_returns) != len(cov_matrix.columns):
        raise ValueError(
            f"Mismatch: expected_returns length ({len(expected_returns)}) != "
            f"cov_matrix size ({len(cov_matrix.columns)})"
        )
    
    if n_scenarios <= 0:
        raise ValueError(f"n_scenarios must be > 0, got {n_scenarios}")
    
    if weeks_ahead <= 0:
        raise ValueError(f"weeks_ahead must be > 0, got {weeks_ahead}")
    
    if random_seed is not None:
        np.random.seed(random_seed)
    
    scenarios = {}
    tickers = list(cov_matrix.columns)
    
    for scenario_idx in range(n_scenarios):
        # Each scenario is weeks_ahead weeks of returns
        scenario_returns = []
        
        # For each week, sample fresh returns from multivariate normal
        # This ensures each week is an independent draw from the distribution
        for week_idx in range(weeks_ahead):
            week_returns = np.random.multivariate_normal(
                mean=expected_returns,
                cov=cov_matrix.values
            )
            scenario_returns.append(week_returns)
        
        # Create dataframe: rows=weeks, columns=tickers
        df_scenario = pd.DataFrame(
            scenario_returns,
            columns=tickers,
            index=range(weeks_ahead)
        )
        df_scenario.index.name = 'week'
        
        scenarios[f"scenario_{scenario_idx}"] = df_scenario
    
    return scenarios

## Testing with Toy Data

In [ ]:
# Create toy data: 3 stocks, simple numbers
toy_returns = [1.02, 1.015, 1.01]  # 2%, 1.5%, 1% geometric growth per week

toy_cov = pd.DataFrame(
    [[0.001, 0.0005, 0.0003],
     [0.0005, 0.0008, 0.0002],
     [0.0003, 0.0002, 0.0006]],
    columns=['STOCK_A', 'STOCK_B', 'STOCK_C']
)

print("Toy covariance matrix:")
print(toy_cov)
print()

# Generate toy scenarios
toy_scenarios = generate_scenarios(
    n_scenarios=3,
    weeks_ahead=5,
    expected_returns=toy_returns,
    cov_matrix=toy_cov,
    random_seed=42
)

print(f"Generated {len(toy_scenarios)} toy scenarios")
print()

# Inspect first scenario
print("First scenario (scenario_0) - 5 weeks of returns for 3 stocks:")
print(toy_scenarios['scenario_0'])
print()
print(f"Shape: {toy_scenarios['scenario_0'].shape}")
print(f"Columns: {list(toy_scenarios['scenario_0'].columns)}")

## Testing with Real Data

In [ ]:
import json

# Load real data from data cleaning output
with open("../../data/expected_returns_covariance_matrix.json") as f:
    data = json.load(f)

real_tickers = data["tickers"]
real_expected_returns = data["expected_returns"]
real_cov_matrix = pd.DataFrame(
    data["covariance_matrix"],
    index=real_tickers,
    columns=real_tickers
)

print(f"Loaded {len(real_tickers)} real stock tickers")
print(f"Expected returns shape: {len(real_expected_returns)}")
print(f"Covariance matrix shape: {real_cov_matrix.shape}")
print()

# Generate real scenarios: 100 scenarios, 52 weeks (1 year)
real_scenarios = generate_scenarios(
    n_scenarios=100,
    weeks_ahead=52,
    expected_returns=real_expected_returns,
    cov_matrix=real_cov_matrix,
    random_seed=123
)

print(f"Generated {len(real_scenarios)} scenarios")
print(f"Each scenario: {real_scenarios['scenario_0'].shape[0]} weeks × {real_scenarios['scenario_0'].shape[1]} stocks")
print()

# Inspect first scenario, first 5 weeks
print("First scenario (scenario_0) - first 5 weeks:")
print(real_scenarios['scenario_0'].head())

## Computing Cumulative Returns per Scenario

For each scenario and ticker, compute the cumulative growth multiplier over the entire forecast period.

In [ ]:
# Compute cumulative returns for each scenario and ticker
cumulative_returns_data = {}

for scenario_key, df_scenario in real_scenarios.items():
    # Extract scenario number from key (e.g., "scenario_0" -> 0)
    scenario_num = int(scenario_key.split('_')[1])
    
    cumulative_returns_data[scenario_num] = {}
    
    # For each ticker, compute cumulative multiplier
    for ticker in df_scenario.columns:
        weekly_returns = df_scenario[ticker].values
        
        # Cumulative multiplier = product of geometric returns
        # (weekly_returns are already in multiplier form, e.g., 1.02, 0.98)
        cumulative_multiplier = np.prod(weekly_returns)
        
        cumulative_returns_data[scenario_num][ticker] = float(cumulative_multiplier)

# Create dataframe for inspection: rows=scenarios, columns=tickers
cumulative_df = pd.DataFrame(cumulative_returns_data).T
cumulative_df.index.name = 'scenario'

print("Cumulative returns (multiplier form):")
print("Rows = scenarios, Columns = tickers")
print()
print(cumulative_df.iloc[:5, :5])  # First 5 scenarios, first 5 tickers
print()
print(f"Shape: {cumulative_df.shape}")
print()

# Convert to JSON format for Pyomo: scenario-first nesting with numeric keys
cumulative_json = {}
for scenario_num in cumulative_returns_data.keys():
    cumulative_json[str(scenario_num)] = cumulative_returns_data[scenario_num]

# Save to file
output_path = "../../data/scenario_cumulative_returns.json"
with open(output_path, "w") as f:
    json.dump(cumulative_json, f, indent=2)

print(f"Saved cumulative returns to {output_path}")
print()
print("JSON structure (first 2 scenarios, first 3 tickers):")
preview = {str(i): {k: cumulative_json[str(i)][k] for k in list(cumulative_json[str(i)].keys())[:3]} 
           for i in range(min(2, len(cumulative_json)))}
print(json.dumps(preview, indent=2))